# Assignment 6: Google Stock Price Prediction
## Recurrent Neural Network (RNN/LSTM) for Time Series Forecasting

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

In [ ]:
# GPU Configuration (Optional)
print("="*60)
print("GPU CONFIGURATION")
print("="*60)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✓ GPU DETECTED: {len(gpus)} GPU(s)")
        print("GPU-accelerated training enabled!")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("\n⚠ NO GPU - Training on CPU")
    print("Note: RNN training on CPU will be slower but functional")

print("="*60)

In [ ]:
# Load dataset
print("\n" + "="*60)
print("LOADING GOOGLE STOCK PRICES DATASET")
print("="*60)

# IMPORTANT: Update the path to your CSV file
# Your file has multi-level columns with Ticker row
data = pd.read_csv('GOOGL.csv', header=[0, 1], index_col=0)

print(f"\nDataset shape: {data.shape}")
print(f"\nColumn structure:")
print(data.columns.tolist())
print(f"\nFirst 5 rows:")
display(data.head())
print(f"\nLast 5 rows:")
display(data.tail())

In [ ]:
# Data preprocessing - Extract Close price
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Extract Close price column (GOOGL ticker)
# Your data has multi-level columns: ('Close', 'GOOGL')
close_prices = data[('Close', 'GOOGL')].copy()

# Convert index to datetime if needed
close_prices.index = pd.to_datetime(close_prices.index)

# Sort by date
close_prices = close_prices.sort_index()

# Remove any NaN values
initial_count = len(close_prices)
close_prices = close_prices.dropna()
final_count = len(close_prices)

print(f"\nData Quality Check:")
print(f"  Initial samples: {initial_count}")
print(f"  After removing NaN: {final_count}")
print(f"  Removed: {initial_count - final_count} rows")

print(f"\nDate Range:")
print(f"  Start date: {close_prices.index.min()}")
print(f"  End date: {close_prices.index.max()}")
print(f"  Total trading days: {len(close_prices)}")

print(f"\nPrice Statistics:")
print(f"  Minimum price: ${close_prices.min():.2f}")
print(f"  Maximum price: ${close_prices.max():.2f}")
print(f"  Mean price: ${close_prices.mean():.2f}")
print(f"  Std deviation: ${close_prices.std():.2f}")

# Convert to numpy array for processing
prices = close_prices.values.reshape(-1, 1)
print(f"\nPrice array shape: {prices.shape}")

In [ ]:
# Visualize stock prices
print("\nVisualizing stock price history...")

plt.figure(figsize=(16, 6))
plt.plot(close_prices.index, close_prices.values, linewidth=2, color='#1f77b4', label='GOOGL Close Price')
plt.xlabel('Date', fontweight='bold', fontsize=12)
plt.ylabel('Stock Price ($)', fontweight='bold', fontsize=12)
plt.title('Google (GOOGL) Stock Price History', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('stock_price_history.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Visualization saved as 'stock_price_history.png'")

In [ ]:
# Feature Scaling (Normalization)
print("\n" + "="*60)
print("FEATURE SCALING")
print("="*60)

print("\nApplying MinMaxScaler to normalize prices to [0, 1] range...")
print("Reason: Neural networks train better with normalized inputs")

# Initialize MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit and transform the data
scaled_prices = scaler.fit_transform(prices)

print(f"\nScaling Results:")
print(f"  Original price range: [${prices.min():.2f}, ${prices.max():.2f}]")
print(f"  Scaled price range: [{scaled_prices.min():.4f}, {scaled_prices.max():.4f}]")
print(f"  Mean (scaled): {scaled_prices.mean():.4f}")
print(f"  Std (scaled): {scaled_prices.std():.4f}")

print(f"\n✓ Feature scaling complete!")

In [ ]:
# Create sequences for time series prediction
print("\n" + "="*60)
print("SEQUENCE CREATION")
print("="*60)

# Use past 60 days to predict next day
time_steps = 60

print(f"\nSequence Configuration:")
print(f"  Lookback window (time steps): {time_steps} days")
print(f"  Prediction horizon: 1 day ahead")
print(f"\nStrategy: Use prices from past 60 days to predict day 61")

def create_sequences(data, time_steps):
    """
    Create input-output sequences for RNN training
    
    Args:
        data: Scaled price data
        time_steps: Number of past days to use for prediction
    
    Returns:
        X: Input sequences [samples, time_steps, features]
        y: Target values [samples]
    """
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i-time_steps:i, 0])  # Past 60 days
        y.append(data[i, 0])               # Day 61 (target)
    return np.array(X), np.array(y)

# Create sequences
X, y = create_sequences(scaled_prices, time_steps)

print(f"\nSequence Creation Results:")
print(f"  Total sequences created: {len(X)}")
print(f"  Input shape (X): {X.shape} [samples, time_steps]")
print(f"  Target shape (y): {y.shape} [samples]")

print(f"\nExample sequence:")
print(f"  Input (first 5 values): {X[0][:5]}")
print(f"  Target: {y[0]:.4f}")

print(f"\n✓ Sequences created successfully!")

In [ ]:
# Train-test split
print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

# Split: 80% train, 20% test (time series - no shuffling!)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"\nSplit Strategy: Temporal split (no shuffling)")
print(f"Reason: Maintain chronological order for time series")

print(f"\nTraining Set:")
print(f"  Sequences: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Shape: {X_train.shape}")

print(f"\nTest Set:")
print(f"  Sequences: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Shape: {X_test.shape}")

# Reshape for RNN: [samples, time_steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"\nReshaped for RNN:")
print(f"  X_train: {X_train.shape} [samples, time_steps, features]")
print(f"  X_test: {X_test.shape}")

print(f"\n✓ Data split and reshaped successfully!")

In [ ]:
# Define hyperparameter configurations
print("\n" + "="*60)
print("HYPERPARAMETER EXPERIMENTATION")
print("="*60)

print("\nExperimenting with THREE key hyperparameters:")
print("  1. RNN Type (LSTM vs GRU)")
print("  2. Number of Units (50 vs 100)")
print("  3. Learning Rate (0.001 vs 0.0001)")
print("  4. Dropout Rate (0.2 vs 0.3)")

configs = [
    # LSTM variations
    {'name': 'Config 1: LSTM, Units=50, LR=0.001, Dropout=0.2', 'rnn_type': 'LSTM', 'units': 50, 'learning_rate': 0.001, 'dropout': 0.2},
    {'name': 'Config 2: LSTM, Units=50, LR=0.001, Dropout=0.3', 'rnn_type': 'LSTM', 'units': 50, 'learning_rate': 0.001, 'dropout': 0.3},
    {'name': 'Config 3: LSTM, Units=100, LR=0.001, Dropout=0.2', 'rnn_type': 'LSTM', 'units': 100, 'learning_rate': 0.001, 'dropout': 0.2},
    
    # Learning rate variation
    {'name': 'Config 4: LSTM, Units=50, LR=0.0001, Dropout=0.2', 'rnn_type': 'LSTM', 'units': 50, 'learning_rate': 0.0001, 'dropout': 0.2},
    
    # GRU variations
    {'name': 'Config 5: GRU, Units=50, LR=0.001, Dropout=0.2', 'rnn_type': 'GRU', 'units': 50, 'learning_rate': 0.001, 'dropout': 0.2},
    {'name': 'Config 6: GRU, Units=100, LR=0.001, Dropout=0.2', 'rnn_type': 'GRU', 'units': 100, 'learning_rate': 0.001, 'dropout': 0.2},
]

print(f"\nTotal configurations: {len(configs)}")

print("\nFixed parameters:")
print("  - Architecture: 2 RNN layers + Dense output")
print("  - Batch size: 32")
print("  - Max epochs: 50 (with early stopping)")
print("  - Early stopping patience: 10 epochs")

print("\nConfigurations to test:")
for i, config in enumerate(configs, 1):
    print(f"  {i}. {config['name']}")

print(f"\nEstimated training time:")
if gpus:
    print("  With GPU: ~15-25 minutes total (~3-4 min per config)")
else:
    print("  With CPU: ~60-90 minutes total (~10-15 min per config)")

In [ ]:
# Train models with different configurations
print("\n" + "="*60)
print("MODEL TRAINING")
print("="*60)
print("\nNote: Using early stopping to prevent overfitting")
print("Training will stop if validation loss doesn't improve for 10 epochs\n")

results = []

for i, config in enumerate(configs, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(configs)}] Training: {config['name']}")
    print(f"{'='*60}")
    
    # Build RNN model
    model = Sequential()
    
    if config['rnn_type'] == 'LSTM':
        # LSTM layers
        model.add(LSTM(units=config['units'], return_sequences=True, input_shape=(X_train.shape[1], 1)))
        model.add(Dropout(config['dropout']))
        model.add(LSTM(units=config['units'], return_sequences=False))
        model.add(Dropout(config['dropout']))
    else:  # GRU
        # GRU layers
        model.add(GRU(units=config['units'], return_sequences=True, input_shape=(X_train.shape[1], 1)))
        model.add(Dropout(config['dropout']))
        model.add(GRU(units=config['units'], return_sequences=False))
        model.add(Dropout(config['dropout']))
    
    # Output layer
    model.add(Dense(1))
    
    # Compile model
    optimizer = Adam(learning_rate=config['learning_rate'])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    print(f"\nModel architecture:")
    model.summary()
    
    # Callbacks
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
    # Train model
    print(f"\nStarting training...")
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )
    
    epochs_trained = len(history.history['loss'])
    
    # Predict on test set
    print(f"\nEvaluating on test set...")
    predictions = model.predict(X_test, verbose=0)
    
    # Inverse transform to get actual prices
    predictions_actual = scaler.inverse_transform(predictions)
    y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    # Calculate metrics
    mse = mean_squared_error(y_test_actual, predictions_actual)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_actual, predictions_actual)
    r2 = r2_score(y_test_actual, predictions_actual)
    
    # Store results
    results.append({
        'Config': config['name'],
        'RNN Type': config['rnn_type'],
        'Units': config['units'],
        'Learning Rate': config['learning_rate'],
        'Dropout': config['dropout'],
        'Epochs': epochs_trained,
        'RMSE': rmse,
        'MAE': mae,
        'R2 Score': r2,
        'Final Train Loss': history.history['loss'][-1],
        'Final Val Loss': history.history['val_loss'][-1]
    })
    
    print(f"\n{'='*60}")
    print(f"RESULTS:")
    print(f"  Epochs trained: {epochs_trained}")
    print(f"  RMSE: ${rmse:.2f}")
    print(f"  MAE: ${mae:.2f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Final train loss: {history.history['loss'][-1]:.6f}")
    print(f"  Final val loss: {history.history['val_loss'][-1]:.6f}")
    print(f"{'='*60}")

print("\n" + "="*60)
print("✓ ALL CONFIGURATIONS TRAINED SUCCESSFULLY!")
print("="*60)

In [ ]:
# Display comprehensive results
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS TABLE")
print("="*60)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values('RMSE')

print("\n(Sorted by RMSE - Lower is Better)\n")
display(results_df_sorted[['Config', 'RNN Type', 'Units', 'Learning Rate', 'Dropout', 
                           'RMSE', 'MAE', 'R2 Score', 'Epochs']])

# Best model
best_config = results_df_sorted.iloc[0]
print("\n" + "="*60)
print("🏆 BEST MODEL")
print("="*60)
print(f"Configuration: {best_config['Config']}")
print(f"\nHyperparameters:")
print(f"  RNN Type: {best_config['RNN Type']}")
print(f"  Units: {int(best_config['Units'])}")
print(f"  Learning Rate: {best_config['Learning Rate']}")
print(f"  Dropout Rate: {best_config['Dropout']}")
print(f"  Epochs Trained: {int(best_config['Epochs'])}")
print(f"\nPerformance Metrics:")
print(f"  RMSE: ${best_config['RMSE']:.2f}")
print(f"  MAE: ${best_config['MAE']:.2f}")
print(f"  R² Score: {best_config['R2 Score']:.4f}")

# Interpretation
if best_config['R2 Score'] > 0.9:
    print(f"\n✓ Excellent prediction accuracy (R² > 0.9)")
elif best_config['R2 Score'] > 0.7:
    print(f"\n✓ Good prediction accuracy (R² > 0.7)")
else:
    print(f"\n⚠ Moderate prediction accuracy (R² < 0.7)")

print(f"\nInterpretation: Model predictions deviate by ~${best_config['MAE']:.2f} on average")
print("="*60)

In [ ]:
# Hyperparameter impact analysis
print("\n" + "="*60)
print("HYPERPARAMETER IMPACT ANALYSIS")
print("="*60)

# RNN Type Impact (LSTM vs GRU)
print("\n1. RNN TYPE IMPACT (LSTM vs GRU):")
print("-" * 60)
rnn_analysis = results_df.groupby('RNN Type').agg({
    'RMSE': 'mean',
    'MAE': 'mean',
    'R2 Score': 'mean',
    'Epochs': 'mean'
}).round(4)
print(rnn_analysis)
best_rnn = rnn_analysis['RMSE'].idxmin()
print(f"\nBest RNN Type: {best_rnn} (Lowest average RMSE)")

# Units Impact
print("\n2. NUMBER OF UNITS IMPACT:")
print("-" * 60)
units_analysis = results_df.groupby('Units').agg({
    'RMSE': 'mean',
    'MAE': 'mean',
    'R2 Score': 'mean'
}).round(4)
print(units_analysis)
best_units = units_analysis['RMSE'].idxmin()
print(f"\nBest Number of Units: {int(best_units)}")

# Learning Rate Impact
print("\n3. LEARNING RATE IMPACT:")
print("-" * 60)
lr_analysis = results_df.groupby('Learning Rate').agg({
    'RMSE': 'mean',
    'MAE': 'mean',
    'R2 Score': 'mean'
}).round(4)
print(lr_analysis)
best_lr = lr_analysis['RMSE'].idxmin()
print(f"\nBest Learning Rate: {best_lr}")

# Dropout Impact
print("\n4. DROPOUT RATE IMPACT:")
print("-" * 60)
dropout_analysis = results_df.groupby('Dropout').agg({
    'RMSE': 'mean',
    'MAE': 'mean',
    'R2 Score': 'mean'
}).round(4)
print(dropout_analysis)
best_dropout = dropout_analysis['RMSE'].idxmin()
print(f"\nBest Dropout Rate: {best_dropout}")

In [ ]:
# Visualize hyperparameter impact
print("\nGenerating hyperparameter analysis visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. RNN Type Comparison
rnn_data = results_df.groupby('RNN Type')['RMSE'].mean()
axes[0, 0].bar(rnn_data.index, rnn_data.values, color=['#FF6B6B', '#4ECDC4'], edgecolor='black', alpha=0.8)
axes[0, 0].set_ylabel('Average RMSE ($)', fontweight='bold', fontsize=12)
axes[0, 0].set_title('LSTM vs GRU Performance', fontweight='bold', fontsize=14)
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(rnn_data.values):
    axes[0, 0].text(i, v, f'${v:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Units Impact
units_data = results_df.groupby('Units')['RMSE'].mean()
axes[0, 1].bar(units_data.index.astype(str), units_data.values, color='#45B7D1', edgecolor='black', alpha=0.8)
axes[0, 1].set_xlabel('Number of Units', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Average RMSE ($)', fontweight='bold', fontsize=12)
axes[0, 1].set_title('Units Impact on RMSE', fontweight='bold', fontsize=14)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(units_data.values):
    axes[0, 1].text(i, v, f'${v:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 3. All Configurations Comparison
axes[1, 0].barh(range(len(results_df_sorted)), results_df_sorted['RMSE'], 
                color='#FFA07A', edgecolor='black', alpha=0.8)
axes[1, 0].set_yticks(range(len(results_df_sorted)))
axes[1, 0].set_yticklabels([f"Config {i+1}" for i in range(len(results_df_sorted))])
axes[1, 0].set_xlabel('RMSE ($)', fontweight='bold', fontsize=12)
axes[1, 0].set_title('RMSE Comparison (All Configs)', fontweight='bold', fontsize=14)
axes[1, 0].grid(axis='x', alpha=0.3)
axes[1, 0].invert_yaxis()
for i, v in enumerate(results_df_sorted['RMSE']):
    axes[1, 0].text(v, i, f' ${v:.2f}', va='center', fontsize=10)

# 4. R² Score Comparison
axes[1, 1].barh(range(len(results_df_sorted)), results_df_sorted['R2 Score'], 
                color='#95E1D3', edgecolor='black', alpha=0.8)
axes[1, 1].set_yticks(range(len(results_df_sorted)))
axes[1, 1].set_yticklabels([f"Config {i+1}" for i in range(len(results_df_sorted))])
axes[1, 1].set_xlabel('R² Score', fontweight='bold', fontsize=12)
axes[1, 1].set_title('R² Score Comparison (All Configs)', fontweight='bold', fontsize=14)
axes[1, 1].grid(axis='x', alpha=0.3)
axes[1, 1].invert_yaxis()
for i, v in enumerate(results_df_sorted['R2 Score']):
    axes[1, 1].text(v, i, f' {v:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('stock_prediction_hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Hyperparameter analysis saved as 'stock_prediction_hyperparameter_analysis.png'")

In [ ]:
# Train final best model and visualize predictions
print("\n" + "="*60)
print("TRAINING FINAL OPTIMIZED MODEL")
print("="*60)

# Get best config details
best_rnn_type = best_config['RNN Type']
best_units_val = int(best_config['Units'])
best_lr_val = best_config['Learning Rate']
best_dropout_val = best_config['Dropout']

print(f"\nOptimal Configuration:")
print(f"  RNN Type: {best_rnn_type}")
print(f"  Units: {best_units_val}")
print(f"  Learning Rate: {best_lr_val}")
print(f"  Dropout: {best_dropout_val}")
print("\nTraining final model...")

# Build final model
final_model = Sequential()
if best_rnn_type == 'LSTM':
    final_model.add(LSTM(units=best_units_val, return_sequences=True, input_shape=(X_train.shape[1], 1)))
    final_model.add(Dropout(best_dropout_val))
    final_model.add(LSTM(units=best_units_val, return_sequences=False))
    final_model.add(Dropout(best_dropout_val))
else:
    final_model.add(GRU(units=best_units_val, return_sequences=True, input_shape=(X_train.shape[1], 1)))
    final_model.add(Dropout(best_dropout_val))
    final_model.add(GRU(units=best_units_val, return_sequences=False))
    final_model.add(Dropout(best_dropout_val))
final_model.add(Dense(1))

optimizer = Adam(learning_rate=best_lr_val)
final_model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
history = final_model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=32, 
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print(f"\n✓ Final model trained successfully!")
print(f"  Epochs: {len(history.history['loss'])}")
print(f"  Final validation loss: {history.history['val_loss'][-1]:.6f}")

# Generate predictions
train_predictions = final_model.predict(X_train, verbose=0)
test_predictions = final_model.predict(X_test, verbose=0)

# Inverse transform to actual prices
train_predictions = scaler.inverse_transform(train_predictions)
test_predictions = scaler.inverse_transform(test_predictions)
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
# Visualize final predictions
print("\nGenerating prediction visualization...")

plt.figure(figsize=(18, 7))

# Plot training data
plt.plot(range(len(y_train_actual)), y_train_actual, 
         label='Training Data (Actual)', color='#1f77b4', linewidth=2, alpha=0.8)

# Plot test data (actual)
test_start = len(y_train_actual)
plt.plot(range(test_start, test_start + len(y_test_actual)), y_test_actual, 
         label='Test Data (Actual)', color='#2ca02c', linewidth=2, alpha=0.8)

# Plot test predictions
plt.plot(range(test_start, test_start + len(test_predictions)), test_predictions, 
         label='Test Data (Predicted)', color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.8)

# Add vertical line separating train and test
plt.axvline(x=test_start, color='red', linestyle=':', linewidth=2, alpha=0.5, label='Train/Test Split')

plt.xlabel('Time (Days)', fontweight='bold', fontsize=13)
plt.ylabel('Stock Price ($)', fontweight='bold', fontsize=13)
plt.title(f'Google Stock Price Prediction - {best_rnn_type} Model\nRMSE: ${best_config["RMSE"]:.2f}, R²: {best_config["R2 Score"]:.4f}', 
          fontweight='bold', fontsize=15)
plt.legend(fontsize=12, loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stock_prediction_final.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Prediction visualization saved as 'stock_prediction_final.png'")

In [ ]:
# Display sample predictions
print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

print("\nLast 10 test predictions:")
print("-" * 60)
print(f"{'Day':<6} {'Actual Price':<15} {'Predicted Price':<18} {'Error':<10}")
print("-" * 60)

for i in range(-10, 0):
    actual = y_test_actual[i][0]
    predicted = test_predictions[i][0]
    error = abs(actual - predicted)
    print(f"{len(y_test_actual)+i:<6} ${actual:<14.2f} ${predicted:<17.2f} ${error:<9.2f}")

print("-" * 60)
avg_error = np.mean(np.abs(y_test_actual - test_predictions))
print(f"\nAverage Prediction Error: ${avg_error:.2f}")
print("="*60)

In [ ]:
# Summary
print("\n" + "="*60)
print("ASSIGNMENT 6 COMPLETE ✓")
print("="*60)

print(f"\nDataset Information:")
print(f"  ✓ Stock: Google (GOOGL)")
print(f"  ✓ Total trading days: {len(close_prices)}")
print(f"  ✓ Training sequences: {len(X_train)}")
print(f"  ✓ Test sequences: {len(X_test)}")
print(f"  ✓ Lookback window: {time_steps} days")

print(f"\nPreprocessing Applied:")
print(f"  ✓ MinMaxScaler normalization [0, 1]")
print(f"  ✓ Sequence creation (60-day lookback)")
print(f"  ✓ Temporal train-test split (80-20)")

print(f"\nHyperparameter Experimentation:")
print(f"  ✓ Tested {len(configs)} configurations")
print(f"  ✓ RNN Types: LSTM, GRU")
print(f"  ✓ Units: 50, 100")
print(f"  ✓ Learning Rates: 0.001, 0.0001")
print(f"  ✓ Dropout Rates: 0.2, 0.3")

print(f"\nBest Model Performance:")
print(f"  ✓ Configuration: {best_config['Config']}")
print(f"  ✓ RMSE: ${best_config['RMSE']:.2f}")
print(f"  ✓ MAE: ${best_config['MAE']:.2f}")
print(f"  ✓ R² Score: {best_config['R2 Score']:.4f}")

print(f"\nKey Findings:")
print(f"  ✓ {best_rnn} outperformed alternative RNN type")
print(f"  ✓ Optimal units: {int(best_units)}")
print(f"  ✓ Optimal learning rate: {best_lr}")
print(f"  ✓ Optimal dropout: {best_dropout}")

print("\n" + "="*60)
print("🎉 TIME SERIES PREDICTION COMPLETE!")
print("="*60)